# Protocolo de Firmas Digitales RSA

En este notebook, mostraremos el proceso para generar una firma digital y su verificación correspondiente usando el protocolo RSA. Empezaremos por cargar las funciones que necesitamos:

In [4]:
import sys
import importlib.util

if 'google.colab' in sys.modules:
    runtime = "Google Colab"
    if importlib.util.find_spec("cryptocalc") is None:
        print("   Installing MA2006B from GitHub\n")
        !pip install git+https://github.com/Krul-dev/MA2006B.git
else:
    runtime = "Local environment"

import cryptocalc

print(f"\n========= Notebook execution context =========")
print(f"              Runtime: {runtime}")
print(f"       Python version: {sys.version.split()[0]}")
print(f"   CryptoCalc version: {cryptocalc.__version__}\n")

from cryptocalc import (
    rsa_key_generation,
    rsa_encryption,
    rsa_decryption,
    sha256_of_sentence,
)


========= Notebook execution context =========
              Runtime: Local environment
       Python version: 3.14.3
   CryptoCalc version: 0.1.0



Una vez que ya hemos cargado las librerías necesarias, empezamos por generar nuestras llaves *pública* y *privada*

In [3]:
(public_key, private_key) = rsa_key_generation(1000)
d = private_key
e = public_key[0]
n = public_key[1]

print(f"Llave privada (d): {d}\n")
print(f"Llave pública (e): {e}\n")
print(f"Módulo para las llaves (n): {n}\n")

Llave privada (d): 45874480181591183690701738940128867796061386912650333425557872356976567906234687323657966787644112490465739882987873531590127844458377157994607724511863797785991317701333752543292627909307474003586574079501005364966285844646166840485037007133602138157483312605802530833363758247763857960959245662373110077565329742277188737185018144030471864517048533146737425871494592704890415340597610968061481037876285620267528672871138752469451183684292247907420466674198798332097054937787477138622508489921504731116862385408525184901667294586225237395912299420488371855964613191572342990765851471009184078983821461

Llave pública (e): 7649412696729430808428152704642865435237768018017749413177850886513494630944750914455423099071072130119322902322876413470676287501841358306035544942218761274537707702666953637562259539101438693729034867316593864783156210664144425258733149191473424036762164177454374370360506140404085411871201424550846257806969518268898630699752086100461580008659925533621958

Recordemos que el valor $d$ de la llave privada se debe de mantener en secreto. En cambio los valores de $e$ y $n$ corresponden a la llave pública y son conocidas por todos los agentes involucrados, incluída Eva.

Para ilustrar el algoritmo de generación de firmas digitales, supongamos que Alicia desea firmar el siguiente mensaje llano $m$:

In [3]:
m = "Hello World!"
h = sha256_of_sentence(m)

print(f"Mensaje llano (m): {m}\n")
print(f"Hash del mensaje llano (h): {h}\n")

Mensaje llano (m): Hello World!

Hash del mensaje llano (h): 57676413081093003148005107550719583540116985236696423860923466490497932824681



Para generar la firma digital $s$, Alicia debe de utilizar el mismo algoritmo que utiliza para descifrar mensajes. En otras palabras, Alicia debe calcular
$$
s = h^{d} \mod n
$$
y ya con esta información puede generar el *mensaje firmado*
$$
\text{mensaje\_firmado} = (m, s, n)
$$

In [4]:
s = rsa_decryption(public_key, private_key, h)
signed_message = (m, s, n)

print(f"El mensaje firmado es: {signed_message}\n")

El mensaje firmado es: ('Hello World!', 638829277311187863056858212801401125078103107833451490251266682587848863184536354555506690404319238919821762104796744132050161330327462100701070669758218575145071086757832643344623847167420692185836557143289660903744884438612509136428035276331215221508645225874244361249615084901228032933275839709222916412008181910130301977930761720568415493433561248067988443648231459092508978409190663097947514264426078708905511146310117235146126761164961338456896091743997578487544787438920786018921437440845240644112477351501838557619521719359698554631758148576944515387224198502053346429710970101270451343820925, 7227208977359639453285200607784026172384327712099373549254694185432735946472868816591995423154235810190072620208573548975740556177749871152022042303310819923010518642923252208868490940095361593748365511004375307691083779221186705542889779009243580240021300723046021055507829016928278631178273300359011420259993353458127793499207185803089537201918584503566674

Para verificar la firma, Beto aplica ahora el mismo algoritmo que utilizaría para encriptar mensajes. De manera más precisa, Beto calcula el Hash
$$
h = \operatorname{Hash}(m)
$$
y calcula también
$$
\tilde{h} = s^{e} \mod n
$$
Si estos dos números son iguales, entonces la firma es válida.

In [5]:
h = sha256_of_sentence(signed_message[0])
s = signed_message[1]
h_tilde = rsa_encryption(public_key, s)

if h == h_tilde:
    verificacion_firma = "la firma es válida"
else:
    verificacion_firma = "la firma es inválida"

print(f"El valor de h es: {h}\n")
print(f"El valor de h̃ es: {h_tilde}\n")
print("Por lo tanto,", verificacion_firma)

El valor de h es: 57676413081093003148005107550719583540116985236696423860923466490497932824681

El valor de h̃ es: 57676413081093003148005107550719583540116985236696423860923466490497932824681

Por lo tanto, la firma es válida
